# Lead Intelligence System - Business Analyst Dashboard

An analyst-facing companion for Workflow Plan Phases 1-6: explore lead data, rubric decisions, LLM explanations, outreach drafts, and report outputs. Run the Phase 1-3 acceptance-check cell below to see current pass/fail status; use TESTING_GUIDE.md for the full manual checklist, end-to-end testing, and submission.

## How to use
1. Run cells from top to bottom with the project environment selected.
2. Start from either the repository root or `lead-qualifier`; setup detects both locations.
3. Run the Phase 1-3 acceptance-check cell to see pass/fail status. Phase 5-6 report previews are rebuilt from the current CSVs and rubric in deterministic mode, so they do not depend on old saved JSON reports.
4. Edit the dataset or reference date in the relevant cells, then rerun dependent cells. Results do not update automatically.
5. The Phase 4 cell makes live LLM requests; skip it for offline analysis and keep API keys in the untracked `.env` file.

**Workflow-plan target:** weighted score 1-10; QUALIFIED >=7.5, REVIEW >=5.0 and <7.5, REJECTED <5.0. Qualified leads receive one of three reproducible, fact-only outreach structures for human review; the notebook never sends it.

**Qualification rules:** missing required name/company fields are routed to REVIEW. Notebook score displays use runtime values directly; the acceptance-check cell reports any configuration or data gaps.

In [ ]:
# ============================================================
# SETUP: Imports and Path Configuration
# ============================================================
import sys
from pathlib import Path

# Resolve paths whether the notebook starts at the repository root or package root.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "lead-qualifier" / "config.yaml").exists():
    PROJECT_ROOT = CURRENT_DIR
    LEAD_QUALIFIER = CURRENT_DIR / "lead-qualifier"
elif (CURRENT_DIR / "config.yaml").exists() and CURRENT_DIR.name == "lead-qualifier":
    LEAD_QUALIFIER = CURRENT_DIR
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Start this notebook from the repository root or lead-qualifier directory.")
sys.path.insert(0, str(LEAD_QUALIFIER))

import pandas as pd
import json
import yaml
from datetime import date, datetime
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv(LEAD_QUALIFIER / ".env")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"Project root: {PROJECT_ROOT}")
print(f"Lead qualifier: {LEAD_QUALIFIER}")
print(f"Exists: {LEAD_QUALIFIER.exists()}")

REFERENCE_DATE = date(2024, 1, 20)

In [ ]:
# ============================================================
# 1. DATA EXPLORATION
# ============================================================

# Load all three datasets
training_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_training.csv")
testing_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_testing.csv")
full_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_full_scale_100.csv")

print("=== DATASET OVERVIEW ===")
print(f"Training:  {len(training_df)} leads")
print(f"Testing:   {len(testing_df)} leads")
print(f"Full:      {len(full_df)} leads")

print("\n=== TRAINING DATA SAMPLE ===")
display(training_df.head(10))

In [ ]:
# Dataset statistics
print("=== TRAINING DATA STATS ===")
print(f"Shape: {training_df.shape}")
print(f"\nColumns: {list(training_df.columns)}")
print(f"\nDtypes:\n{training_df.dtypes}")
print(f"\nMissing values:\n{training_df.isnull().sum()}")
print(f"\nCompany size stats:\n{training_df['company_size'].describe()}")
print(f"\nIndustry distribution (top 10):\n{training_df['industry'].value_counts().head(10)}")
print(f"\nSource distribution:\n{training_df['source'].value_counts()}")
print(f"\nDate range: {training_df['last_interaction_date'].min()} to {training_df['last_interaction_date'].max()}")

In [ ]:
# Testing data - edge cases
print("=== TESTING DATA - EDGE CASES ===")
print(f"Shape: {testing_df.shape}")
print(f"\nMissing values:\n{testing_df.isnull().sum()}")
print(f"\nCompany size unique (including NaN):\n{testing_df['company_size'].value_counts(dropna=False)}")
print(f"\nIndustry unique:\n{testing_df['industry'].value_counts(dropna=False)}")
print(f"\nSource unique:\n{testing_df['source'].value_counts(dropna=False)}")

# Show rows with missing data
edge_cases = testing_df[testing_df.isnull().any(axis=1)]
print(f"\nRows with missing data ({len(edge_cases)}):")
display(edge_cases)

In [ ]:
# Full scale data overview
print("=== FULL SCALE DATA (100 leads) ===")
print(f"Shape: {full_df.shape}")
print(f"\nMissing values:\n{full_df.isnull().sum()}")
print(f"\nCompany size stats:\n{full_df['company_size'].describe()}")
print(f"\nIndustry count: {full_df['industry'].nunique()} unique")
print(f"\nSource distribution:\n{full_df['source'].value_counts()}")
print(f"\nDate range: {full_df['last_interaction_date'].min()} to {full_df['last_interaction_date'].max()}")

In [ ]:
# ============================================================
# 2. RUBRIC CONFIGURATION VIEWER
# ============================================================

with open(LEAD_QUALIFIER / "config.yaml") as f:
    config = yaml.safe_load(f)

rubric = config["rubric"]
raw_thresholds = config["thresholds"]
thresholds = raw_thresholds

def plan_score(value):
    """Display the runtime score on the workflow plan's 1-10 scale."""
    return round(float(value), 1)

llm_config = config["llm"]

print("=== RUBRIC FACTORS & WEIGHTS ===")
rubric_summary = []
for name, factor in rubric.items():
    rubric_summary.append({
        "Factor": name.replace("_", " ").title(),
        "Weight": f"{factor['weight']*100:.0f}%",
        "Tiers": len(factor["tiers"]),
        "Default": factor["tiers"][-1].get("default", "N/A")
    })
display(pd.DataFrame(rubric_summary))
print(f"Plan thresholds (1-10): qualified >= {thresholds['qualified']}, review >= {thresholds['review']}")
print(f"Configured thresholds: {raw_thresholds}")
if raw_thresholds.get("qualified") != 7.5 or raw_thresholds.get("review") != 5.0:
    print("WARNING: runtime thresholds do not match WORKFLOW_PLAN.md.")

In [ ]:
# Detailed tier breakdown for each factor
for name, factor in rubric.items():
    print(f"\n=== {name.replace('_', ' ').title()} (Weight: {factor['weight']*100:.0f}%) ===")
    tiers_data = []
    for i, tier in enumerate(factor["tiers"]):
        if "range" in tier:
            desc = f"{tier['range'][0]} - {tier['range'][1]} employees"
        elif "days" in tier:
            desc = f"{tier['days'][0]} - {tier['days'][1]} days ago"
        elif "industries" in tier:
            desc = ", ".join(tier["industries"])
        elif "sources" in tier:
            desc = ", ".join(tier["sources"])
        elif "condition" in tier:
            desc = f"Condition: {tier['condition'][:80]}..."
        else:
            desc = "Default"
        tiers_data.append({
            "Tier": i+1,
            "Criteria": desc,
            "Score": tier.get("score", tier.get("default", "N/A"))
        })
    display(pd.DataFrame(tiers_data))

In [ ]:
print("=== DECISION THRESHOLDS ===")
threshold_df = pd.DataFrame([
    {"Decision": "QUALIFIED", "Min Score": thresholds["qualified"], "Max Score": 10.0},
    {"Decision": "REVIEW", "Min Score": thresholds["review"], "Max Score": round(thresholds["qualified"] - 0.1, 1)},
    {"Decision": "REJECTED", "Min Score": 0.0, "Max Score": round(thresholds["review"] - 0.1, 1)}
])
display(threshold_df)

print("=== LLM CONFIGURATION ===")
display(pd.DataFrame([llm_config]).T.rename(columns={0: "Value"}))

In [ ]:
# ============================================================
# PHASE 1-3 ACCEPTANCE CHECKS (NO LLM CALLS)
# ============================================================

from importlib.util import find_spec
from utils.data_loader import load_leads, validate_lead
from utils.rubric_engine import load_rubric, score_lead, score_leads_batch
from utils.models import LeadResult
from utils.report_builder import build_report

phase_checks = []

def record_check(phase, check, passed, evidence):
    phase_checks.append({
        "Phase": phase,
        "Check": check,
        "Status": "PASS" if passed else "FAIL",
        "Evidence": evidence
    })

# Phase 1: required project files, dependencies, and secret exclusion.
required_files = [
    "main.py", "config.yaml", "requirements.txt", "README.md", ".env.example",
    ".gitignore", "leads.csv", "utils/__init__.py", "utils/models.py",
    "utils/data_loader.py", "utils/rubric_engine.py", "utils/llm_client.py",
    "utils/message_generator.py", "utils/report_builder.py", "tests/__init__.py",
    "tests/test_rubric.py", "tests/test_data_validation.py", "tests/test_edge_cases.py",
    "tests/test_phase4_5.py", "tests/test_phase6_reports.py"
]
missing_files = [name for name in required_files if not (LEAD_QUALIFIER / name).is_file()]
record_check("1", "Required project files", not missing_files, f"Missing: {missing_files or 'none'}")

required_modules = {"pandas": "pandas", "PyYAML": "yaml", "requests": "requests", "tenacity": "tenacity", "python-dotenv": "dotenv", "groq": "groq"}
missing_modules = [package for package, module in required_modules.items() if find_spec(module) is None]
record_check("1", "Planned dependencies available", not missing_modules, f"Missing: {missing_modules or 'none'}")

ignore_rules = (LEAD_QUALIFIER / ".gitignore").read_text(encoding="utf-8").splitlines()
env_ignored = any(rule.strip() == ".env" for rule in ignore_rules)
record_check("1", ".env ignore rule", env_ignored, ".env is listed in the package .gitignore" if env_ignored else ".env ignore rule not found")

# Phase 2: rubric structure, configured thresholds, and actual score scale.
weight_total = sum(factor["weight"] for factor in rubric.values())
record_check("2", "3-5 factors with weights totaling 1.0", 3 <= len(rubric) <= 5 and abs(weight_total - 1.0) < 1e-9, f"{len(rubric)} factors; weight total={weight_total:.3f}")

plan_thresholds_match = raw_thresholds.get("qualified") == 7.5 and raw_thresholds.get("review") == 5.0
record_check("2", "Plan thresholds (7.5 / 5.0)", plan_thresholds_match, f"Configured values: {raw_thresholds}")

runtime_rubric = load_rubric(str(LEAD_QUALIFIER / "config.yaml"))
training_leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))
testing_leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_testing.csv"))
full_leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_full_scale_100.csv"))
sample_score = score_lead(training_leads[0], runtime_rubric, reference_date=REFERENCE_DATE)
record_check("2", "Actual score is on 1-10 scale", 1 <= sample_score.total_score <= 10, f"First training lead score={sample_score.total_score}; factors={len(sample_score.factor_scores)}")

# Phase 3: dataset sizes, deterministic edge-case processing, and report visibility.
record_check("3", "Training data has at least 30 leads", len(training_leads) >= 30, f"Observed {len(training_leads)} leads")
record_check("3", "Testing and full-scale dataset counts", len(testing_leads) > 0 and len(full_leads) == 100, f"Testing={len(testing_leads)}; full-scale={len(full_leads)}")

testing_scores = score_leads_batch(testing_leads, runtime_rubric, reference_date=REFERENCE_DATE)
record_check("3", "Testing rows load and score without an exception", len(testing_scores) == len(testing_leads), f"Scored {len(testing_scores)}/{len(testing_leads)} rows")

testing_results = [LeadResult(lead=score.lead, score=score) for score in testing_scores]
testing_report = build_report(testing_results, {"reference_date": REFERENCE_DATE.isoformat()})
invalid_required = [lead for lead in testing_leads if {"name", "company"}.intersection(validate_lead(lead))]
reported_names = {item["name"] for item in testing_report["flagged_for_review"]}
reported_names.update(item["lead"] for item in testing_report["errors"])
unreported_required = [lead.name or "<missing name>" for lead in invalid_required if lead.name not in reported_names]
record_check("3", "Missing required fields appear in review/errors", not unreported_required, f"Unreported required-field records: {unreported_required or 'none'}")

def lead_key(item):
    return (item.get("name", ""), item.get("company", ""))

review_keys = {lead_key(item) for item in testing_report["leads"] if item["decision"] == "REVIEW"}
flagged_keys = {lead_key(item) for item in testing_report["flagged_for_review"]}
record_check("3", "All REVIEW decisions are flagged", review_keys.issubset(flagged_keys), f"Unflagged review records: {sorted(review_keys - flagged_keys)}")

qualified_keys = {lead_key(item) for item in testing_report["leads"] if item["decision"] == "QUALIFIED"}
queue_keys = {lead_key(item) for item in testing_report["priority_queue"]}
record_check("3", "Priority queue contains qualified leads only", queue_keys == qualified_keys, f"Queue entries={len(queue_keys)}; qualified={len(qualified_keys)}")

display(pd.DataFrame(phase_checks))

In [ ]:
# ============================================================
# 3. SCORING ENGINE DEMO - Single Lead Scoring Explorer
# ============================================================

from utils.data_loader import load_leads
from utils.rubric_engine import load_rubric, score_lead
from utils.models import Lead

rubric = load_rubric(str(LEAD_QUALIFIER / "config.yaml"))

def score_single_lead(lead_data, reference_date=REFERENCE_DATE):
    """Score a single lead and return detailed breakdown."""
    lead = Lead.from_dict(lead_data)
    score = score_lead(lead, rubric, reference_date)
    
    result = {
        "lead": {
            "name": lead.name,
            "company": lead.company,
            "company_size": lead.company_size,
            "industry": lead.industry,
            "source": lead.source,
            "last_interaction_date": str(lead.last_interaction_date) if lead.last_interaction_date else None
        },
        "total_score": plan_score(score.total_score),
        "decision": score.decision,
        "priority": score.priority,
        "factor_breakdown": {}
    }
    for fs in score.factor_scores:
        result["factor_breakdown"][fs.name] = {
            "raw_score": fs.raw_score,
            "weight": fs.weight,
            "weighted_score": round(fs.weighted_score, 2),
            "contribution_pct": round(fs.weighted_score / sum(f.weighted_score for f in score.factor_scores) * 100, 1)
        }
    return result

# Demo: Score first 3 training leads
leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))

for lead in leads[:3]:
    result = score_single_lead({
        "name": lead.name,
        "company": lead.company,
        "company_size": lead.company_size,
        "industry": lead.industry,
        "source": lead.source,
        "last_interaction_date": lead.last_interaction_date
    })
    print(f"\n{'='*60}")
    print(f"{result['lead']['name']} - {result['lead']['company']}")
    print(f"Total Score: {result['total_score']} | Decision: {result['decision']} | Priority: {result['priority']}")
    print(f"Factor Breakdown:")
    for fname, fdata in result["factor_breakdown"].items():
        print(f"  {fname}: raw={fdata['raw_score']}, weighted={fdata['weighted_score']}, contrib={fdata['contribution_pct']}%")

In [ ]:
# ============================================================
# 4. BATCH SCORING RESULTS - Training Data
# ============================================================

from utils.rubric_engine import score_leads_batch

leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))
scores = score_leads_batch(leads, rubric, reference_date=REFERENCE_DATE)

# Convert to DataFrame for analysis
results = []
for s in scores:
    row = {
        "name": s.lead.name,
        "company": s.lead.company,
        "company_size": s.lead.company_size,
        "industry": s.lead.industry,
        "source": s.lead.source,
        "last_interaction_date": s.lead.last_interaction_date,
        "total_score": plan_score(s.total_score),
        "decision": s.decision,
        "priority": s.priority
    }
    for fs in s.factor_scores:
        row[f"factor_{fs.name}"] = fs.raw_score
    results.append(row)

results_df = pd.DataFrame(results)

print("=== TRAINING DATA - SCORING SUMMARY ===")
print(f"Total leads: {len(results_df)}")
print(f"\nDecision distribution:")
print(results_df['decision'].value_counts())
print(f"\nPriority distribution:")
print(results_df['priority'].value_counts())
print(f"\nScore statistics:")
print(results_df['total_score'].describe())

display(results_df[['name', 'company', 'total_score', 'decision', 'priority']].head(15))

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Score distribution
axes[0, 0].hist(results_df['total_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=thresholds["qualified"], color='green', linestyle='--', label=f'Qualified ({thresholds["qualified"]:.1f})')
axes[0, 0].axvline(x=thresholds["review"], color='orange', linestyle='--', label=f'Review ({thresholds["review"]:.1f})')
axes[0, 0].set_xlabel('Total Score (1-10)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Score Distribution (Training)')
axes[0, 0].legend()

# Decision pie
decision_counts = results_df['decision'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0, 1].pie(decision_counts.values, labels=decision_counts.index, autopct='%1.1f%%', colors=colors)
axes[0, 1].set_title('Decision Distribution')

# Score by industry (top 10)
industry_scores = results_df.groupby('industry')['total_score'].mean().sort_values(ascending=False).head(10)
industry_scores.plot(kind='barh', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_xlabel('Avg Score')
axes[1, 0].set_title('Avg Score by Industry (Top 10)')
axes[1, 0].invert_yaxis()

# Score by source
source_scores = results_df.groupby('source')['total_score'].mean().sort_values(ascending=False)
source_scores.plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_ylabel('Avg Score')
axes[1, 1].set_title('Avg Score by Source')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Factor contribution analysis
factor_cols = [c for c in results_df.columns if c.startswith('factor_')]
factor_means = results_df[factor_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
factor_means.plot(kind='bar', color='mediumpurple')
plt.title('Average Raw Factor Scores (1-10 scale)')
plt.ylabel('Average Raw Score')
plt.xlabel('Factor')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Average factor scores:")
for col, val in factor_means.items():
    print(f"  {col}: {val:.2f}")

In [ ]:
# ============================================================
# 5. OUTPUT REPORT VIEWER
# ============================================================

# Rebuild report previews from the current source data and rubric; no saved reports or LLM calls are used.
from utils.models import LeadResult
from utils.report_builder import build_report

def build_dashboard_report(dataset_path):
    report_leads = load_leads(str(dataset_path))
    report_scores = score_leads_batch(report_leads, rubric, reference_date=REFERENCE_DATE)
    report_results = [LeadResult(lead=score.lead, score=score) for score in report_scores]
    return build_report(report_results, {
        "processed_at": date.today().isoformat(),
        "source_file": dataset_path.name,
        "reference_date": REFERENCE_DATE.isoformat(),
        "processing_mode": "deterministic_no_llm"
    })

training_report = build_dashboard_report(PROJECT_ROOT / "leads_training.csv")
testing_report = build_dashboard_report(PROJECT_ROOT / "leads_testing.csv")
full_report = build_dashboard_report(PROJECT_ROOT / "leads_full_scale_100.csv")

print("=== REPORT METADATA ===")
for name, report in [("Training", training_report), ("Testing", testing_report), ("Full", full_report)]:
    meta = report["metadata"]
    print(f"\n{name} Report:")
    for k, v in meta.items():
        print(f"  {k}: {v}")

In [ ]:
print("=== SUMMARY COMPARISON ===")
summary_data = []
for name, report in [("Training", training_report), ("Testing", testing_report), ("Full", full_report)]:
    s = report["summary"]
    summary_data.append({
        "Dataset": name,
        "Total": s["total_processed"],
        "Qualified": s["qualified_count"],
        "Review": s["review_count"],
        "Rejected": s["rejected_count"],
        "Qualified %": s["qualified_percentage"],
        "Flagged": len(report["flagged_for_review"]),
        "Errors": len(report["errors"])
    })
display(pd.DataFrame(summary_data))

In [ ]:
# Full report - priority queue (top 20)
print("=== FULL SCALE - PRIORITY QUEUE (TOP 20) ===")
pq_df = pd.DataFrame(full_report["priority_queue"])
if not pq_df.empty and "score" in pq_df:
    pq_df["score"] = pq_df["score"].map(plan_score)
display(pq_df.head(20))
non_qualified_queue = pq_df[pq_df["priority"] != "HIGH"] if not pq_df.empty else pd.DataFrame()
print(f"Non-qualified leads in priority_queue (plan expects 0): {len(non_qualified_queue)}")

In [ ]:
# Full report - sample outreach messages
print("=== SAMPLE OUTREACH MESSAGES ===")
for msg in full_report["sample_outreach_messages"]:
    print(f"\n--- {msg['lead']} ---")
    print(msg['message'][:300])
    print("...")

In [ ]:
# Full report - flagged for review
print("=== FLAGGED FOR REVIEW ===")
if full_report["flagged_for_review"]:
    flagged_df = pd.DataFrame(full_report["flagged_for_review"])
    display(flagged_df)
else:
    print("No leads flagged in the full-scale report.")

# Testing report - flagged for review
print("\n=== TESTING REPORT - FLAGGED FOR REVIEW ===")
if testing_report["flagged_for_review"]:
    flagged_df = pd.DataFrame(testing_report["flagged_for_review"])
    display(flagged_df)

review_names = {item.get("name") for item in full_report["leads"] if item.get("decision") == "REVIEW"}
flagged_names = {item.get("name") for item in full_report["flagged_for_review"]}
unflagged_review = review_names - flagged_names
print(f"Review decisions missing from flagged_for_review (plan expects 0): {len(unflagged_review)}")
if unflagged_review:
    print(sorted(unflagged_review))

In [ ]:
# ============================================================
# 6. FILTERING & EXPLORATION
# ============================================================

# Full scale leads detail
full_leads_df = pd.DataFrame(full_report["leads"])
if "score" in full_leads_df:
    full_leads_df["score"] = full_leads_df["score"].map(plan_score)

print("=== FILTER EXAMPLE: QUALIFIED LEADS ===")
print(f"Total leads: {len(full_leads_df)}")
print(f"Columns: {list(full_leads_df.columns)}")

# Filter examples
print("\n--- Qualified Leads ---")
qualified = full_leads_df[full_leads_df['decision'] == 'QUALIFIED']
print(f"Count: {len(qualified)}")
display(qualified[['name', 'company', 'company_size', 'industry', 'source', 'score', 'priority']].head(15))

In [ ]:
# Filter by industry
print("--- Leads by Industry (Qualified only) ---")
if len(qualified) > 0:
    industry_qual = qualified['industry'].value_counts()
    display(industry_qual.to_frame('Qualified Count'))

In [ ]:
# Filter by company size range
print("--- Score Distribution by Company Size Bands ---")
full_leads_df['size_band'] = pd.cut(full_leads_df['company_size'], 
    bins=[0, 10, 50, 500, 5000, 10000, float('inf')],
    labels=['1-10', '11-50', '51-500', '501-5000', '5001-10000', '10000+'])

size_band_stats = full_leads_df.groupby('size_band', observed=False).agg(
    count=('score', 'count'),
    avg_score=('score', 'mean'),
    qualified=('decision', lambda x: (x == 'QUALIFIED').sum())
).round(1)
display(size_band_stats)

In [ ]:
# Factor breakdown for qualified vs rejected
print("=== FACTOR SCORES: QUALIFIED vs REJECTED ===")

if "factor_breakdown" in full_leads_df.columns:
    for factor_name in rubric:
        column = f"factor_{factor_name}"
        full_leads_df[column] = full_leads_df["factor_breakdown"].map(
            lambda breakdown: breakdown.get(factor_name, {}).get("raw") if isinstance(breakdown, dict) else None
        )
factor_cols = [f"factor_{factor_name}" for factor_name in rubric if f"factor_{factor_name}" in full_leads_df.columns]
if factor_cols:
    qualified_factors = full_leads_df[full_leads_df['decision'] == 'QUALIFIED'][factor_cols].mean()
    rejected_factors = full_leads_df[full_leads_df['decision'] == 'REJECTED'][factor_cols].mean()
    review_factors = full_leads_df[full_leads_df['decision'] == 'REVIEW'][factor_cols].mean()
    
    comparison = pd.DataFrame({
        'Qualified': qualified_factors,
        'Review': review_factors,
        'Rejected': rejected_factors
    }).round(2)
    display(comparison)
    
    # Plot
    comparison.plot(kind='bar', figsize=(10, 5))
    plt.title('Average Factor Scores by Decision')
    plt.ylabel('Raw Score (1-10)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 3. DATA-QUALITY AND EDGE-CASE REVIEW
# ============================================================

print("=== TESTING DATA - EDGE CASE SCORING ===")

testing_leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_testing.csv"))
testing_scores = score_leads_batch(testing_leads, rubric, reference_date=REFERENCE_DATE)

for s in testing_scores:
    issues = []
    if s.lead.company_size is None: issues.append("No size")
    if not s.lead.industry: issues.append("No industry")
    if not s.lead.source: issues.append("No source")
    if s.lead.last_interaction_date is None: issues.append("No date")
    
    if issues:
        print(f"\n{s.lead.name} ({s.lead.company}):")
        print(f"  Issues: {', '.join(issues)}")
        print(f"  Score: {plan_score(s.total_score)} / 10 | Decision: {s.decision} | Priority: {s.priority}")
        for fs in s.factor_scores:
            print(f"  {fs.name}: raw={fs.raw_score}, weighted={fs.weighted_score:.2f}")

# ============================================================
# 4. LLM INTEGRATION - PHASE 4
# ============================================================

This optional cell makes live batched requests and requires the configured provider package, API key in `.env`, and network access. The mocked unit tests work offline.

The deterministic rubric remains authoritative for score, decision, and priority. The LLM may add only an explanation and human review note, matched by stable per-run lead ID. Malformed or partial responses are retried with strict JSON instructions; remaining failures are reported per lead.

---


In [ ]:
# ============================================================
# 4. LLM INTEGRATION - PHASE 4
# ============================================================

print("=== PHASE 4: LLM INTEGRATION ===\n")

# Import LLM client
from utils.llm_client import enrich_leads_with_llm

# Load configuration
with open(LEAD_QUALIFIER / "config.yaml") as f:
    config = yaml.safe_load(f)

print(f"Provider: {config['llm']['provider']} | Model: {config['llm']['model']} | Batch size: {config['llm']['batch_size']}")

# Load leads and rubric
leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))
rubric = load_rubric(str(LEAD_QUALIFIER / "config.yaml"))
baseline_scores = score_leads_batch(leads, rubric, reference_date=REFERENCE_DATE)

print(f"Loading {len(leads)} leads for LLM enrichment...\n")

# Enrich explanations while passing the deterministic scores into the prompt.
enrichments = enrich_leads_with_llm(leads, config, lead_scores=baseline_scores)

successful_count = sum(not item["error"] for item in enrichments)
print(f"LLM responses: {successful_count}/{len(enrichments)} successful\n")

# Keep displayed scores/decisions from the rubric; LLM output has no scoring fields.
results = []
enrichments_by_id = {item["lead_id"]: item for item in enrichments}
for index, score in enumerate(baseline_scores):
    expected_id = f"lead-{index:06d}"
    enrichment = enrichments_by_id[expected_id]
    results.append({
        "lead_id": expected_id,
        "name": score.lead.name,
        "company": score.lead.company,
        "total_score": plan_score(score.total_score),
        "decision": score.decision,
        "priority": score.priority,
        "llm_reasoning": enrichment["reasoning"],
        "llm_review_note": enrichment["review_note"],
        "error": enrichment["error"],
    })

llm_results_df = pd.DataFrame(results)

print("=== LLM Results Summary ===")
print(f"Total enriched: {len(llm_results_df)}")
print(f"Leads with enrichment errors: {llm_results_df['error'].notna().sum()}")

# Show first 5 leads with LLM reasoning
print("=== Sample LLM Reasoning ===")
for _, row in llm_results_df.head(5).iterrows():
    print(f"  {row['name']} ({row['company']}):")
    print(f"    Rubric: {row['total_score']} / 10 | {row['decision']} | {row['priority']}")
    reason = row['llm_reasoning'][:150] if row['llm_reasoning'] else "No reasoning"
    print(f"    LLM: {reason}...")
    print()

In [ ]:
# Visualize LLM-enriched results
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Decision distribution
decision_counts = llm_results_df["decision"].value_counts()
axes[0].pie(decision_counts.values, labels=decision_counts.index, autopct="%1.1f%%",
            colors=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0].set_title("Decision Distribution (LLM-Enriched)")

# Score distribution
axes[1].hist(llm_results_df["total_score"], bins=20, edgecolor="black", alpha=0.7)
axes[1].axvline(x=thresholds["qualified"], color="green", linestyle="--", label=f"Qualified (>= {thresholds['qualified']:.1f})")
axes[1].axvline(x=thresholds["review"], color="orange", linestyle="--", label=f"Review (>= {thresholds['review']:.1f})")
axes[1].set_xlabel("Total Score (1-10)")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution (LLM-Enriched)")
axes[1].legend()

plt.tight_layout()
plt.show()

print("=== Phase 4 Complete ===")
print("Scores, decisions, and priorities shown above come from the deterministic rubric.")
print("Next: Phase 5 - Message Generation & Report Builder")

# ============================================================
# 5. MESSAGE GENERATION & REPORT BUILDER - PHASE 5
# ============================================================

This phase reviews three reproducible first-touch structures and the output report.
Drafts are generated only for QUALIFIED leads. Variant selection uses the stable report lead ID; message facts come only from the supplied company, industry, employee count, and source. No customer needs, budgets, or product claims are inferred. Drafts are for human review and are never sent by the application.

---


In [ ]:
# ============================================================
# 5. MESSAGE GENERATION & REPORT BUILDER - PHASE 5
# ============================================================

print("=== PHASE 5: MESSAGE GENERATION & REPORT BUILDER ===\n")

# Import message generator and report builder
from utils.message_generator import generate_outreach, OUTREACH_VARIANTS, get_outreach_variant_id
from utils.report_builder import build_report, build_summary_report
from utils.models import LeadResult

print("Outreach variants:", [variant_id for variant_id, _ in OUTREACH_VARIANTS])
print("Plan requirement: drafts for QUALIFIED leads only; salesperson review; never sent automatically.")

# Load leads and scores
leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))
rubric = load_rubric(str(LEAD_QUALIFIER / "config.yaml"))
scores = score_leads_batch(leads, rubric, reference_date=REFERENCE_DATE)

# Create LeadResult objects
lead_results = [LeadResult(lead=ls.lead, score=ls, llm_reasoning=None, outreach_message=None, error=None) for ls in scores]

# Build report with messages
metadata = {"processed_at": datetime.now().astimezone().isoformat(timespec="seconds"), "total_leads": len(leads), "rubric_version": "1.0", "llm_model": "deterministic-only"}
report = build_report(lead_results, metadata)

print(f"Report built with {len(report['sample_outreach_messages'])} sample messages")
print(f"Priority queue: {len(report['priority_queue'])} leads")
print(f"Flagged for review: {len(report['flagged_for_review'])}")

# Confirm examples only belong to qualified leads, as required by the plan.
qualified_names = {ls.lead.name for ls in scores if ls.decision == "QUALIFIED"}
draft_names = {item["lead"] for item in report["sample_outreach_messages"]}
print(f"Non-qualified sample drafts (plan expects 0): {len(draft_names - qualified_names)}")
assert all((item["outreach_message"] is not None) == (item["decision"] == "QUALIFIED") for item in report["leads"])
assert not (draft_names - qualified_names)
print(f"Qualified leads with drafts: {sum(item['outreach_message'] is not None for item in report['leads'])}")
assert all((item.get("outreach_variant") in {variant_id for variant_id, _ in OUTREACH_VARIANTS}) == (item["decision"] == "QUALIFIED") for item in report["leads"])
leads_by_id = {f"lead-{index:06d}": ls.lead for index, ls in enumerate(scores)}
assert all(item["outreach_variant"] == get_outreach_variant_id(leads_by_id[item["lead_id"]], item["lead_id"]) for item in report["sample_outreach_messages"])

# Show sample qualified messages
print("\n=== SAMPLE OUTREACH MESSAGES ===")
for msg in report["sample_outreach_messages"]:
    print(f"  {msg['lead']}:")
    print(f"    {msg['message'][:150]}...")
    print()

In [ ]:
# Inspect the three distinct, fact-only first-touch structures
print("=== QUALIFIED OUTREACH VARIANTS ===")
for variant_id, template in OUTREACH_VARIANTS:
    print(f"\n--- {variant_id} ---")
    print(template)

In [ ]:
# Full report visualization
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Decision distribution
decision_counts = pd.DataFrame(report["leads"])["decision"].value_counts()
axes[0, 0].pie(decision_counts.values, labels=decision_counts.index, autopct="%1.1f%%",
             colors=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0, 0].set_title("Decision Distribution")

# 2. Priority distribution
priority_counts = pd.DataFrame(report["leads"])["priority"].value_counts()
axes[0, 1].pie(priority_counts.values, labels=priority_counts.index, autopct="%1.1f%%",
             colors=["#e74c3c", "#f39c12", "#2ecc71"])
axes[0, 1].set_title("Priority Distribution")

# 3. Score distribution with messages
leads_df = pd.DataFrame(report["leads"])
if "score" in leads_df:
    leads_df["score"] = leads_df["score"].map(plan_score)
axes[1, 0].hist(leads_df["score"], bins=20, edgecolor="black", alpha=0.7)
axes[1, 0].axvline(x=thresholds["qualified"], color="green", linestyle="--", label=f"Qualified (>= {thresholds['qualified']:.1f})")
axes[1, 0].axvline(x=thresholds["review"], color="orange", linestyle="--", label=f"Review (>= {thresholds['review']:.1f})")
axes[1, 0].set_xlabel("Total Score (1-10)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Score Distribution")
axes[1, 0].legend()

# 4. Messages by priority
msg_priority = pd.DataFrame(report["sample_outreach_messages"])
if len(msg_priority) > 0:
    msg_priority_counts = msg_priority["priority"].value_counts()
    msg_priority_counts.plot(kind="bar", ax=axes[1, 1], color=["#e74c3c", "#f39c12", "#2ecc71"])
    axes[1, 1].set_title("Sample Qualified Outreach Drafts")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("=== Phase 5 Complete ===")
print("Message generation and report builder integrated successfully!")
print("Next: Phase 6 - Report Generation and Validation")

In [ ]:
# ============================================================
# 6. REPORT EXPORT & SHARING
# ============================================================

# Export full results to CSV for Excel/BI tools
full_leads_df.to_csv(LEAD_QUALIFIER / "full_scale_results.csv", index=False)
results_df.to_csv(LEAD_QUALIFIER / "training_results.csv", index=False)

print("Exported:")
print(f"  - {LEAD_QUALIFIER / 'full_scale_results.csv'}")
print(f"  - {LEAD_QUALIFIER / 'training_results.csv'}")

# Export summary for presentation
summary_export = {
    "project": "Lead Intelligence System",
    "phase": "Analyst dashboard review",
    "rubric": {name: {"weight": f["weight"], "tiers": len(f["tiers"])} for name, f in rubric.items()},
    "thresholds": thresholds,
    "training_results": training_report["summary"],
    "testing_results": testing_report["summary"],
    "full_scale_results": full_report["summary"],
    "top_10_leads": full_report["priority_queue"][:10]
}

with open(LEAD_QUALIFIER / "phase3_summary.json", "w") as f:
    json.dump(summary_export, f, indent=2, default=str)

print(f"  - {LEAD_QUALIFIER / 'phase3_summary.json'}")

print("Dashboard complete. Use exported CSVs in Excel, Tableau, or Power BI.")

In [ ]:
# ============================================================
# PHASES 6-9 ACCEPTANCE CHECKS (OFFLINE; NO API KEY REQUIRED)
# ============================================================
import ast, json, subprocess, sys, tempfile, time, zipfile
from datetime import datetime
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "lead-qualifier" / "main.py").is_file():
    root, package = cwd, cwd / "lead-qualifier"
elif (cwd / "main.py").is_file() and (cwd / "config.yaml").is_file():
    package, root = cwd, cwd.parent
else:
    raise FileNotFoundError("Start from repository root or lead-qualifier.")

def verify(label, passed, details):
    print(f"{'PASS' if passed else 'FAIL'}: {label} - {details}")
    if not passed:
        raise AssertionError(label)

project_python = package / "venv" / "Scripts" / "python.exe"
if not project_python.is_file():
    project_python = package / "venv" / "bin" / "python"
if not project_python.is_file():
    project_python = Path(sys.executable)
dependency_check = subprocess.run([str(project_python), "-c", "import pandas, yaml, requests, tenacity, dotenv"], capture_output=True, text=True)
verify("project dependencies", dependency_check.returncode == 0, (dependency_check.stdout + dependency_check.stderr)[-300:])

with tempfile.TemporaryDirectory(prefix="lead-qualifier-analyst-") as temp_dir:
    test_run = subprocess.run([str(project_python), "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=package, capture_output=True, text=True, timeout=300)
    verify("TESTING_GUIDE unit tests", test_run.returncode == 0, (test_run.stdout + test_run.stderr)[-500:])
    reports, runtimes = {}, {}
    for dataset, count in (("training", 30), ("testing", 20), ("full_scale_100", 100)):
        output = Path(temp_dir) / f"{dataset}.json"
        start = time.perf_counter()
        run = subprocess.run([str(project_python), "main.py", "--input", str(root / f"leads_{dataset}.csv"), "--output", str(output), "--reference-date", "2024-01-20", "--no-llm"], cwd=package, capture_output=True, text=True, timeout=300)
        runtimes[dataset] = time.perf_counter() - start
        verify(f"{dataset} CLI run", run.returncode == 0, (run.stdout + run.stderr)[-300:])
        reports[dataset] = json.loads(output.read_text(encoding="utf-8"))
        verify(f"{dataset} lead count", len(reports[dataset]["leads"]) == count, str(len(reports[dataset]["leads"])))

required = {"metadata", "summary", "leads", "priority_queue", "sample_outreach_messages", "flagged_for_review", "errors"}
for dataset, report in reports.items():
    rows, summary = report["leads"], report["summary"]
    verify(f"{dataset} report sections", required <= report.keys(), str(sorted(report)))
    metadata = report["metadata"]
    metadata_valid = all(key in metadata for key in ("processed_at", "total_leads", "rubric_version", "llm_model")) and metadata["total_leads"] == len(rows) and datetime.fromisoformat(metadata["processed_at"]).tzinfo is not None and metadata["llm_model"] == "deterministic-only"
    verify(f"{dataset} metadata", metadata_valid, str(metadata))
    verify(f"{dataset} deterministic reasoning", all({"lead_id", "score", "decision", "priority", "priority_rank", "reasoning", "factor_breakdown", "outreach_message"} <= row.keys() and 1 <= row["score"] <= 10 and row["reasoning"] for row in rows), f"{len(rows)} rows")
    variant_ids = {"fact_first", "question_first", "direct_context"}
    verify(f"{dataset} qualified outreach variants", all((row.get("outreach_variant") in variant_ids) == (row["decision"] == "QUALIFIED") for row in rows), f"{sum(row.get('outreach_variant') in variant_ids for row in rows)} qualified variants")
    ranks = {row["lead_id"]: row["priority_rank"] for row in rows}
    verify(f"{dataset} unique sequential ranks", len(set(ranks.values())) == len(rows) and set(ranks.values()) == set(range(1, len(rows) + 1)), f"{len(ranks)} ranked leads")
    summary_valid = summary["total_processed"] == len(rows) and summary["qualified_count"] + summary["review_count"] + summary["rejected_count"] == len(rows) and summary["qualified_percentage"] == round(100 * summary["qualified_count"] / len(rows), 1) and isinstance(summary["common_rejection_reasons"], list)
    verify(f"{dataset} summary totals", summary_valid, str(summary))
    qualified_ids = {row["lead_id"] for row in rows if row["decision"] == "QUALIFIED"}
    queue = report["priority_queue"]
    verify(f"{dataset} qualified queue", {row["lead_id"] for row in queue} == qualified_ids and all(row["priority"] == "HIGH" and row["priority_rank"] == ranks[row["lead_id"]] for row in queue) and [row["score"] for row in queue] == sorted((row["score"] for row in queue), reverse=True) and [row["priority_rank"] for row in queue] == sorted(row["priority_rank"] for row in queue), f"{len(queue)} queue records")
    samples = report["sample_outreach_messages"]
    rows_by_id = {row["lead_id"]: row for row in rows}
    sample_facts_valid = all(item["lead"] in item["message"] and item["company"] in item["message"] and (not item["industry"] or item["industry"] in item["message"]) and (not item["source"] or item["source"] in item["message"]) and item["outreach_variant"] == rows_by_id[item["lead_id"]].get("outreach_variant") for item in samples)
    verify(f"{dataset} qualified-only fact-grounded drafts", all((row["outreach_message"] is not None) == (row["decision"] == "QUALIFIED") for row in rows) and len(samples) <= 5 and (summary["qualified_count"] < 3 or len(samples) >= 3) and all(row["lead_id"] in qualified_ids for row in samples) and len({row["message"] for row in samples}) == len(samples) and sample_facts_valid, f"{len(samples)} varied samples")
    flagged_ids = {row["lead_id"] for row in report["flagged_for_review"]}
    review_ids = {row["lead_id"] for row in rows if row["decision"] == "REVIEW"}
    verify(f"{dataset} review/errors visibility", review_ids <= flagged_ids and isinstance(report["errors"], list), f"unflagged={len(review_ids - flagged_ids)}; errors={len(report['errors'])}")

missing_company = next((row for row in reports["testing"]["leads"] if row["name"] == "Test_MissingCompany"), None)
verify("Phase 7 missing-company edge case", missing_company is not None and missing_company["decision"] == "REVIEW", str(missing_company))
verify("Phase 7 full-scale runtime", runtimes["full_scale_100"] < 300, f"{runtimes['full_scale_100']:.2f}s; deterministic/offline")

readme = (package / "README.md").read_text(encoding="utf-8")
guide = (root / "TESTING_GUIDE.md").read_text(encoding="utf-8")
readme_lower = readme.lower()
verify("Phase 8 README", all(term in readme_lower for term in ("quick start", "qualification rubric", "known limitations")) and "directional" in readme_lower and len(readme.split()) <= 450, "required sections, concise length, and limitations present")
verify("Phase 8 testing guide", all(f"Phase {phase}" in guide for phase in (6, 7, 8)), "Phases 6-8 documented")

sources = [package / "main.py", *(package / "utils").glob("*.py")]
missing_annotations = []
for source in sources:
    for node in ast.walk(ast.parse(source.read_text(encoding="utf-8"))):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            params = (*node.args.posonlyargs, *node.args.args, *node.args.kwonlyargs)
            if node.returns is None or any(arg.arg not in {"self", "cls"} and arg.annotation is None for arg in params):
                missing_annotations.append(f"{source.name}:{node.name}")
verify("Phase 8 application type annotations", not missing_annotations, str(missing_annotations or "complete"))

final_report_path = package / "output_report.json"
final_report = json.loads(final_report_path.read_text(encoding="utf-8")) if final_report_path.is_file() else {}
final_rows = final_report.get("leads", [])
final_model = final_report.get("metadata", {}).get("llm_model")
final_ids = {row["lead_id"] for row in final_rows}
final_rows_by_id = {row["lead_id"]: row for row in final_rows}
offline_rows = {row["lead_id"]: row for row in reports["full_scale_100"]["leads"]}
final_scores_preserved = final_ids == set(offline_rows) and all((row["score"], row["decision"], row["priority"]) == (offline_rows[row["lead_id"]]["score"], offline_rows[row["lead_id"]]["decision"], offline_rows[row["lead_id"]]["priority"]) for row in final_rows)
final_live_valid = final_model == "deterministic-only" or (all(isinstance(row.get("llm_reasoning"), str) and row["llm_reasoning"].strip() for row in final_rows) and not final_report.get("errors"))
final_variant_ids = {"fact_first", "question_first", "direct_context"}
final_variants_valid = all((row.get("outreach_variant") in final_variant_ids) == (row["decision"] == "QUALIFIED") for row in final_rows)
final_samples_valid = all(item.get("outreach_variant") == final_rows_by_id.get(item["lead_id"], {}).get("outreach_variant") and item["company"] in item["message"] and item["lead"] in item["message"] and (not item.get("industry") or item["industry"] in item["message"]) and (not item.get("source") or item["source"] in item["message"]) for item in final_report.get("sample_outreach_messages", []))
final_ranks = {row["lead_id"]: row.get("priority_rank") for row in final_rows}
final_queue_valid = all(row.get("priority_rank") == final_ranks.get(row["lead_id"]) for row in final_report.get("priority_queue", []))
final_report_valid = len(final_rows) == 100 and final_report.get("metadata", {}).get("total_leads") == 100 and sorted(final_ranks.values()) == list(range(1, 101)) and final_queue_valid and final_scores_preserved and final_live_valid and final_variants_valid and final_samples_valid
verify("Phase 9 final 100-lead report", final_report_valid, f"model={final_model}; reasoning={sum(bool(row.get('llm_reasoning')) for row in final_rows)}/100; errors={len(final_report.get('errors', []))}; deterministic_match={final_scores_preserved}; variants={final_variants_valid}")
archive_path = root / "lead-qualifier-submission.zip"
archive_required = {"lead-qualifier/main.py", "lead-qualifier/output_report.json", "lead-qualifier/README.md", "lead-qualifier/requirements.txt", "lead_analyst_dashboard.ipynb", "TESTING_GUIDE.md", "WORKFLOW_PLAN.md", "leads_training.csv", "leads_testing.csv", "leads_full_scale_100.csv"}
if archive_path.is_file():
    with zipfile.ZipFile(archive_path) as archive:
        archive_entries = set(archive.namelist())
        archive_safe = not any(".env" == Path(name).name or "venv" in Path(name).parts or "__pycache__" in Path(name).parts or name.endswith(".pyc") for name in archive_entries)
        archive_valid = archive.testzip() is None and archive_required <= archive_entries and archive_safe and archive.read("lead-qualifier/output_report.json") == final_report_path.read_bytes()
else:
    archive_valid = False
verify("Phase 9 submission archive", archive_valid, f"exists={archive_path.is_file()}; missing={sorted(archive_required - (archive_entries if archive_path.is_file() else set()))}")
print("Offline Phases 6-9 acceptance checks complete. Run the separate Optional live Groq/API smoke test cell for a bounded authenticated-provider check.")

## Optional live Groq/API smoke test

Before running, revoke any Groq key pasted into a chat or notebook and put a newly rotated key in `lead-qualifier/.env` as `GROQ_API_KEY=...`. Never put the key in notebook source. The cell checks the authenticated `/models` endpoint, then runs the actual CLI against five training rows in two batches and confirms the LLM did not change rubric scores or decisions. It does not print the key or API response body.

**Latest verified result (2026-10-30): PASS.** `/models` returned HTTP 200; the configured model was available; all five leads were enriched in two batches with no report errors; deterministic scores, decisions, and priorities were unchanged. This smoke test does not certify 100-lead live-provider throughput. Run the code cell below to repeat the check.

In [ ]:
# Optional live Groq smoke test; requires a rotated GROQ_API_KEY in lead-qualifier/.env.
import csv, json, subprocess, sys, tempfile
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "lead-qualifier" / "main.py").is_file():
    root, package = cwd, cwd / "lead-qualifier"
elif (cwd / "main.py").is_file() and (cwd / "config.yaml").is_file():
    package, root = cwd, cwd.parent
else:
    raise FileNotFoundError("Start from repository root or lead-qualifier.")

project_python = package / "venv" / "Scripts" / "python.exe"
if not project_python.is_file():
    project_python = package / "venv" / "bin" / "python"
if not project_python.is_file():
    project_python = Path(sys.executable)

# The key is loaded in a child process and is never returned to the notebook.
models_probe = '''
import os, sys
from pathlib import Path
import requests, yaml
from dotenv import load_dotenv
load_dotenv(Path(".env"))
key = os.getenv("GROQ_API_KEY")
if not key:
    print("GROQ_API_KEY is missing from the environment and lead-qualifier/.env.")
    raise SystemExit(2)
try:
    response = requests.get("https://api.groq.com/openai/v1/models", headers={"Authorization": f"Bearer {key}"}, timeout=20)
except requests.RequestException as exc:
    print(f"Groq models endpoint could not be reached ({type(exc).__name__}).")
    raise SystemExit(3)
print(f"Groq models endpoint HTTP status: {response.status_code}")
if response.status_code != 200:
    print("Response body suppressed.")
    raise SystemExit(4)
try:
    model_ids = {item.get("id") for item in response.json().get("data", [])}
    configured_model = yaml.safe_load(Path("config.yaml").read_text(encoding="utf-8"))["llm"]["model"]
except (ValueError, KeyError, TypeError, AttributeError):
    print("Groq returned an unexpected model-list response or config.yaml model.")
    raise SystemExit(5)
available = configured_model in model_ids
print(f"Configured model available: {available} ({configured_model})")
if not available:
    raise SystemExit(6)
'''
probe = subprocess.run([str(project_python), "-c", models_probe], cwd=package, capture_output=True, text=True, timeout=30)
print(probe.stdout.strip())
if probe.returncode != 0:
    raise RuntimeError("Live Groq preflight failed. Check the status above, rotate/set GROQ_API_KEY, and confirm the configured model.")

training_path = root / "leads_training.csv"
with training_path.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    fields = reader.fieldnames
    smoke_leads = [next(reader) for _ in range(5)]
if not fields or len(smoke_leads) != 5:
    raise RuntimeError("The training CSV must contain at least five leads.")

with tempfile.TemporaryDirectory(prefix="groq-live-smoke-") as temp_dir:
    temp = Path(temp_dir)
    input_path, deterministic_path, llm_path = temp / "five_leads.csv", temp / "deterministic.json", temp / "llm.json"
    with input_path.open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=fields)
        writer.writeheader()
        writer.writerows(smoke_leads)
    common = [str(project_python), "main.py", "--input", str(input_path), "--reference-date", "2024-01-20"]
    deterministic_run = subprocess.run(common + ["--output", str(deterministic_path), "--no-llm"], cwd=package, capture_output=True, text=True, timeout=90)
    if deterministic_run.returncode != 0:
        raise RuntimeError("The deterministic baseline CLI run failed.")
    llm_run = subprocess.run(common + ["--output", str(llm_path), "--batch-size", "3", "--verbose"], cwd=package, capture_output=True, text=True, timeout=180)
    if llm_run.returncode != 0 or not llm_path.is_file():
        raise RuntimeError(f"Live CLI run failed (exit code {llm_run.returncode}); captured output was suppressed.")
    baseline = json.loads(deterministic_path.read_text(encoding="utf-8"))
    live_report = json.loads(llm_path.read_text(encoding="utf-8"))
    baseline_rows, live_rows = baseline["leads"], live_report["leads"]
    unchanged = len(live_rows) == 5 and all((a["score"], a["decision"], a["priority"]) == (b["score"], b["decision"], b["priority"]) for a, b in zip(baseline_rows, live_rows))
    enriched = sum(bool(row.get("llm_reasoning")) for row in live_rows)
    error_count = len(live_report.get("errors", []))
    two_batches = llm_run.stderr.count("Processing LLM batch") == 2
    print(f"Live CLI results: {enriched}/5 explanations; report errors: {error_count}; two batches observed: {two_batches}; rubric outputs unchanged: {unchanged}")
    if error_count or enriched != 5 or not two_batches or not unchanged:
        raise AssertionError("Live Groq acceptance failed; do not share API keys or raw response bodies when troubleshooting.")
    print("PASS: authenticated Groq access, configured model, two-batch enrichment, and deterministic-score preservation.")